In [0]:
display(spark.read.table("samples.accuweather.forecast_hourly_metric").limit(5))
display(spark.read.table("samples.accuweather.forecast_daily_calendar_metric").limit(5))
display(spark.read.table("samples.accuweather.historical_daily_calendar_metric").limit(5))

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS weather_project;

CREATE SCHEMA IF NOT EXISTS weather_project.bronze;
CREATE SCHEMA IF NOT EXISTS weather_project.silver;
CREATE SCHEMA IF NOT EXISTS weather_project.gold;
CREATE SCHEMA IF NOT EXISTS weather_project.monitoring;


In [0]:
from pyspark.sql.functions import current_timestamp

source = "samples.accuweather"
target = "weather_project.bronze"

tables = {
    "hourly":           f"{source}.forecast_hourly_metric",
    "daily_forecast":   f"{source}.forecast_daily_calendar_metric",
    "daily_historical": f"{source}.historical_daily_calendar_metric",
}

for target_name, source_table in tables.items():
    df = spark.table(source_table)
    df = df.withColumn("ingestion_ts", current_timestamp())

    df.write.mode("overwrite").saveAsTable(f"{target}.{target_name}")

    print(f" {source_table} ---> {target}.{target_name} ({df.count()} rows)")
                                           


In [0]:

# проверяем корректность создания таблиц и добавления ingestion_ts
display(spark.read.table("weather_project.bronze.hourly").limit(5))
display(spark.read.table("weather_project.bronze.daily_forecast").limit(5))
display(spark.read.table("weather_project.bronze.daily_historical").limit(5))